In [1]:
model = "llama3.2:1b"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [2]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = ChatOllama(model=model)

In [3]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="bge-m3")

In [4]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection_name = "ai_model_book"
collection = client.get_collection(name=collection_name)

# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embedding_model)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = prompt | model | StrOutputParser()

In [6]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(
    query=search_query,
    k=4 
)

print(docs)

[Document(metadata={'page': 33, 'source': './AI_Book.pdf'}, page_content='Types of Machine Learning Systems\nThere are so many different types of Machine Learning systems that it is useful to\nclassify them in broad categories based on:\n Whether or not they are trained with human supervision (supervised, unsuper\nvised, semisupervised, and Reinforcement Learning)\n Whether or not they can learn incrementally on the fly (online versus batch\nlearning)\n Whether they work by simply comparing new data points to known data points,\nor instead detect patterns in the training data and build a predictive model, much\nlike scientists do (instance-based versus model-based learning)\nThese criteria are not exclusive; you can combine them in any way you like. For\nexample, a state-of-the-art spam filter may learn on the fly using a deep neural net\nwork model trained using examples of spam and ham; this makes it an online, model-\nbased, supervised learning system.\nLets look at each of these cr

In [7]:
summary = chain.invoke({"docs": docs})
print(summary)

Based on the retrieved documents, the main themes that can be summarized are:

1. **Classification of Machine Learning Systems**: The documents discuss how different types of machine learning systems can be classified based on their criteria such as supervised/unsupervised learning, incremental learning, pattern detection, and instance-based vs model-based learning.
2. **Supervision and Training Data**: The documents highlight the importance of supervision during training, categorizing it into four main categories: supervised, unsupervised, semisupervised, and Reinforcement Learning.
3. **Types of Machine Learning Systems**: The documents explain that machine learning systems can be categorized into various types based on their design, such as supervised or model-based learning, instance-based vs model-based learning, online vs batch learning, and so on.
4. **Data Gathering and Analysis**: The documents emphasize the importance of gathering relevant data in a training set to feed to a 

In [8]:
# Simple stream the chain output
for chunk in chain.stream(summary):
    print(chunk, end="", flush=True)

Based on the retrieved documents, the main themes that can be summarized are:

1. **Machine Learning System Classification**: The importance of categorizing different types of machine learning systems based on their characteristics, such as supervised/unsupervised learning, incremental learning, pattern detection, and instance-based vs model-based learning.
2. **Supervision and Training Data**: The crucial role of supervision during training in distinguishing between various categories of machine learning systems, including four main types: supervised, unsupervised, semisupervised, and Reinforcement Learning.
3. **Machine Learning System Variability**: The diversity of design characteristics among different types of machine learning systems, such as instance-based vs model-based learning, online vs batch learning, and more.
4. **Data Analysis and Preprocessing**: The necessity of collecting relevant data in a training set to feed to a learning algorithm and analyzing this data to impro

In [9]:
# More complex async event streaming
async for event in chain.astream_events(summary, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

C:\Users\Dominik\AppData\Local\Temp\ipykernel_10140\4172408297.py:2: LangChainBetaWarning: This API is in beta and may change in the future.
  async for event in chain.astream_events(summary, version="v2"):


The main themes retrieved from the documents are:

1. **Classification of Machine Learning Systems**: The importance of categorizing different types of machine learning systems based on their characteristics.
2. **Supervision and Training Data**: The significance of supervision during training, as well as the importance of gathering relevant data in a training set to feed into a learning algorithm.
3. **Types of Machine Learning Systems**: The variety of design approaches, such as supervised or model-based learning, instance-based vs model-based learning, online vs batch learning, and more.
4. **Data Gathering and Analysis**: The crucial role of acquiring and analyzing relevant data in training machine learning models to achieve predictions and improve performance.

These themes collectively suggest that understanding the complexities of machine learning systems is essential for developing effective machine learning models.

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [10]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever(
    search_kwargs={"k": 4}
)

# ADD HERE YOUR CODE
qa_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

In [11]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000024022F50050>, search_kwargs={'k': 4})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"))])
| ChatOllama(model='llama3.2:1b', _client=<ollama._client.Client object at 0x0000024013728950>, _async_client=<ollama._client.AsyncClient object at 0x00000240139F1ED0>)
| StrOutputParser()

In [12]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(question)

'Supervised learning is one of four major categories of machine learning systems, where the training data includes desired solutions (labels) and the algorithm learns from this labeled data to predict outcomes or make decisions.'

In [13]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the training data includes desired solutions, called labels, which the algorithm aims to predict or output. This approach involves training a model on labeled data, and then using it for tasks such as classification, regression, or clustering.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [14]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

In [15]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [16]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [17]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where you have labeled data, and an algorithm tries to learn patterns or relationships between inputs (features) and outputs (labels). In this context, the "teacher" labels are provided by humans or the system itself.

The goal of supervised learning is for the algorithm to make predictions or classify new instances based on the learned patterns. The training data typically includes a combination of input features and corresponding labels.

Here's an example:

Let's say you have a dataset with images of cats and dogs, along with their respective labels (e.g., cat or dog). A supervised learning algorithm is trained on this data to learn how to classify new images as either "cat" or "dog".

The algorithm uses the labeled data to learn the relationships between features (e.g., color, size, shape) and labels. For instance, it might learn that:

* Cats are typically larger than dogs
* Cats have pointy ears
* Dogs have floppy ears

With this 

In [18]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Here's the rephrased standalone question:

Which machine learning algorithms can be used for supervised learning?The following machine learning algorithms are commonly used for supervised learning:

1. k-Nearest Neighbors (kNN)
2. Linear Regression
3. Logistic Regression
4. Support Vector Machines (SVMs)

These algorithms are all types of supervised classification or regression algorithms that learn from labeled data to make predictions or decisions based on input data.